### LLM : 도메인 특화 챗봇 개발 파이프라인

- 이 노트북은 **`도메인 특화 챗봇` 개발 파이프라인 예제 실습**을 수행하는 **Google Colab용** 노트북입니다.

##### Llama를 활용한 도메인 특화 챗봇 개발 파이프라인 예제
   1.  한국 민사법 도메인 특화 LLM Fine-Tuning 및 성능 평가 (Cell 7개)

### 한국 민사법 도메인 특화 LLM Fine-Tuning 및 성능 평가

사전학습된 Llama 3.2 Korean Bllossom 모델(`Bllossom/llama-3.2-Korean-Bllossom-AICA-5B`)을 AI Hub 민사법 데이터셋으로 Fine-tuning합니다.

법률 전문 AI 어시스턴트 구축을 위해 데이터 전처리부터 모델 학습, 평가까지 전체 파이프라인을 구현하여 한국 법률 질의응답 성능을 향상시킵니다.

* AI Hub 민사법 JSON 데이터를 로드하고 법률 텍스트 특화 전처리 (조항 번호 정규화, 날짜 형식 통일 등) 수행
* 4비트 양자화로 메모리 효율적인 모델 로딩 및 LoRA를 통한 파라미터 효율적 학습 설정
* Llama 프롬프트 템플릿 적용하여 시스템-사용자-어시스턴트 대화 형식으로 데이터 구성
* SFTTrainer를 사용하여 답변 부분만 선택적으로 학습하는 Supervised Fine-tuning 실행
* ROUGE 스코어 및 법률 용어 정확도 측정으로 다각도 성능 평가 수행

In [1]:
!pip install -q transformers==4.52.4
!pip install -q peft==0.12.0
!pip install -q trl==0.18.2
!pip install -q datasets
!pip install -q bitsandbytes==0.46.0
!pip install -q accelerate
!pip install -q rouge-score
!pip install -q sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 12.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# ============================================
# Cell 1: 라이브러리 임포트 및 환경 설정
# ============================================

# 필수 라이브러리 임포트
import torch
import transformers
import peft
import os
import warnings
warnings.filterwarnings('ignore')

# PyTorch 및 CUDA 환경 정보 출력
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"GPU 이름: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"Transformers 버전: {transformers.__version__}")
print(f"PEFT 버전: {peft.__version__}")

# GPU 연산 속도 최적화를 위한 설정
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # 비동기 CUDA 실행 활성화
torch.backends.cuda.matmul.allow_tf32 = True  # TensorFloat-32 연산 허용
torch.backends.cudnn.allow_tf32 = True  # cuDNN에서 TF32 사용
torch.backends.cudnn.benchmark = True  # 최적의 알고리즘 자동 선택

PyTorch 버전: 2.9.0+cu126
CUDA 사용 가능: True
GPU 이름: Tesla T4
GPU 메모리: 14.74 GB
Transformers 버전: 4.52.4
PEFT 버전: 0.12.0


In [3]:
# ============================================
# Cell 2: 데이터 로드 및 전처리 함수 정의
# ============================================

import json
import re
from datasets import Dataset as HFDataset
from google.colab import drive

def create_sample_legal_data(num_samples=100):
    """
    실습용 샘플 법률 데이터 생성
    실제 민사법 판결문과 법령 기반 질의응답 데이터
    """
    sample_qa_pairs = [
        # 판결문 기반 질의응답
        {
            "question": "골프 경기 중 선수의 공이 빗나가 다른 사람에게 상해를 입혔을 때 선수에게 요구되는 주의의무는 무엇인가요?",
            "answer": "골프 경기에서 타구를 하는 선수는 자신의 공이 빗나갈 경우를 포함하여 공이 날아갈 것으로 예상되는 범위 내에 사람이 있는지를 확인해야 합니다. 만약 사람이 있는 경우, 목표 방향으로 정확히 보낼 수 있다는 특별한 사정이 없는 이상, 그 사람이 안전한 곳으로 이동할 때까지 기다리거나 공이 사람에게 날아가지 않는 방향으로 타구해야 할 주의의무가 있습니다.",
            "context": "골프와 같은 개인 운동경기에 참가하는 자는 자신의 행동으로 인해 다른 사람이 다칠 수도 있어서 경기 규칙을 준수하고 주위를 살펴 상해의 결과가 발생하는 것을 미연에 방지해야 할 주의의무가 있습니다."
        },
        {
            "question": "피보험자가 입원하지 않았음에도 불구하고 입원확인서를 발급받아 보험금을 청구할 경우 어떤 법적 책임이 따르나요?",
            "answer": "피보험자가 실제로 입원하지 않았음에도 불구하고 허위의 입원확인서를 발급받아 보험금을 청구한 경우, 이는 보험사기를 구성하며, 이는 형사처벌의 대상이 될 수 있습니다. 또한 손해배상 청구 소송에서 보험사에 대한 불법행위로 인정되어, 피보험자는 그로 인해 발생한 손해를 배상할 책임이 있습니다.",
            "context": "피고들은 6시간 이상 입원치료를 받은 사실이 없음에도, 원고로부터 6시간 이상 입원하여 치료를 받았다는 취지로 된 허위의 입원확인서를 발급받아, 이를 삼성화재해상보험 주식회사에 제출하며 보험금을 청구하였습니다."
        },
        {
            "question": "의료행위로 인한 결과 발생 시 의사의 무과실 입증 책임이 요구되는 경우도 있나요?",
            "answer": "의료행위 도중 환자에게 최종 결과의 원인이 된 증상이 발생한 경우, 의료상의 과실 이외의 다른 원인이 있다고 보기 어려운 간접사실들이 입증되면 그 증상이 의료상의 과실에 기한 것이라고 추정할 수 있습니다. 하지만 의사의 과실로 인한 결과 발생을 추정할 수 있을 정도의 개연성이 담보되지 않은 사정들이 있을 때 막연하게 중한 결과에서 의사의 과실과 인과관계를 추정하며 의사에게 무과실의 입증 책임을 지우는 것은 허용되지 않습니다.",
            "context": "의료행위는 고도의 전문적 지식을 필요로 하는 분야로서 전문가가 아닌 일반인으로서는 의사의 의료행위의 과정에 주의의무 위반이 있는지의 여부나 그 주의의무 위반과 손해발생 사이에 인과관계가 있는지 여부를 밝혀내기가 극히 어려운 특수성이 있습니다."
        },
        # 법령 기반 질의응답
        {
            "question": "민사소송에서 당사자의 정당한 이유 없는 결석 시의 조치는 무엇인가요?",
            "answer": "민사소송법 제268조에 따르면, 양쪽 당사자가 변론기일에 출석하지 않거나 출석하였으나 변론하지 않으면 재판장은 새로운 변론기일을 정하여 통지해야 하고, 새로운 변론기일에도 출석하지 않으면 소를 취하한 것으로 본다는 규정이 있습니다.",
            "context": "민사소송법 제268조(양 쪽 당사자가 출석하지 아니한 경우) ①양 쪽 당사자가 변론기일에 출석하지 아니하거나 출석하였다 하더라도 변론하지 아니한 때에는 재판장은 다시 변론기일을 정하여 양 쪽 당사자에게 통지하여야 한다."
        },
        {
            "question": "민사소송법상 보통재판적은 어떻게 정해지나요?",
            "answer": "민사소송법 제2조와 제3조에 따르면, 소는 피고의 보통재판적이 있는 곳의 법원이 관할하며, 사람의 보통재판적은 그의 주소에 따라 정합니다. 대한민국에 주소가 없거나 주소를 알 수 없는 경우에는 거소에 따라 정하고, 거소가 일정하지 아니하거나 거소도 알 수 없으면 마지막 주소에 따라 정합니다.",
            "context": "민사소송법 제2조(보통재판적) 소(訴)는 피고의 보통재판적(普通裁判籍)이 있는 곳의 법원이 관할한다. 제3조(사람의 보통재판적) 사람의 보통재판적은 그의 주소에 따라 정한다."
        },
        {
            "question": "민사소송에서 증인이 정당한 사유 없이 출석하지 않은 경우 어떤 제재가 있나요?",
            "answer": "민사소송법 제311조에 따르면, 증인이 정당한 사유 없이 출석하지 아니한 때에 법원은 결정으로 증인에게 이로 말미암은 소송비용을 부담하도록 명하고 500만원 이하의 과태료에 처합니다. 또한 과태료 재판을 받고도 정당한 사유 없이 다시 출석하지 아니한 때에는 법원은 결정으로 증인을 7일 이내의 감치에 처할 수 있습니다.",
            "context": "민사소송법 제311조(증인이 출석하지 아니한 경우의 과태료 등) ①증인이 정당한 사유 없이 출석하지 아니한 때에 법원은 결정으로 증인에게 이로 말미암은 소송비용을 부담하도록 명하고 500만원 이하의 과태료에 처한다."
        },
        {
            "question": "계약 해제와 해지의 차이점은 무엇인가요?",
            "answer": "계약 해제는 계약을 소급적으로 무효화하는 것으로, 계약이 처음부터 없었던 것으로 되돌리는 효과가 있습니다. 반면 계약 해지는 장래에 대해서만 효력을 상실시키는 것으로, 해지 시점까지의 계약 효력은 유지되고 그 이후부터만 계약이 종료됩니다.",
            "context": "민법 제543조에 따르면 계약 해제는 당사자 일방이 채무를 이행하지 아니한 때에 상대방이 상당한 기간을 정하여 그 이행을 최고하고 그 기간 내에 이행하지 아니한 때에 할 수 있습니다."
        },
        {
            "question": "의사의 설명의무란 무엇이며 어떤 경우에 필요한가요?",
            "answer": "의사는 환자에게 시술을 시행하는 과정 및 그 후에 나쁜 결과가 발생할 개연성이 있는 의료행위를 하는 경우, 응급환자의 경우나 그 밖의 특별한 사정이 없는 한, 당해 환자나 법정대리인에게 질병의 증상, 치료 방법의 내용 및 필요성, 발생이 예상되는 위험 등에 관하여 당시의 의료수준에 비추어 상당하다고 생각되는 사항을 설명하여야 합니다.",
            "context": "의사의 설명의무는 그 의료행위에 따르는 후유증이나 부작용 등의 위험발생 가능성이 희소하다는 사정만으로 면제될 수 없으며, 그 후유증이나 부작용이 당해 치료행위에 전형적으로 발생하는 위험이거나 회복할 수 없는 중대한 것인 경우에는 설명의 대상이 됩니다."
        },
        {
            "question": "공동불법행위자들 간의 구상권은 어떻게 행사되나요?",
            "answer": "공동불법행위자들은 피해자에 대한 관계에서는 연대책임을 지되, 내부관계에서는 과실의 정도에 따라 일정한 부담 부분이 있습니다. 공동불법행위자 중 1인이 자기의 부담 부분 이상을 변제하여 공동의 면책을 얻게 하였을 때에는 다른 공동불법행위자에게 그 부담 부분의 비율에 따라 구상권을 행사할 수 있습니다.",
            "context": "공동불법행위자는 채권자에 대한 관계에서는 연대책임(부진정연대채무)을 지되, 공동불법행위자들 내부관계에서는 일정한 부담 부분이 있고, 이 부담 부분은 공동불법행위자의 과실의 정도에 따라 정하여집니다."
        },
        {
            "question": "민사소송에서 화해권고결정이란 무엇인가요?",
            "answer": "민사소송법 제225조에 따르면, 법원은 소송에 계속중인 사건에 대하여 직권으로 당사자의 이익, 그 밖의 모든 사정을 참작하여 청구의 취지에 어긋나지 아니하는 범위안에서 사건의 공평한 해결을 위한 화해권고결정을 할 수 있습니다. 당사자가 2주 이내에 이의신청을 하지 않으면 재판상 화해와 같은 효력을 가집니다.",
            "context": "민사소송법 제225조(결정에 의한 화해권고) ①법원ㆍ수명법관 또는 수탁판사는 소송에 계속중인 사건에 대하여 직권으로 당사자의 이익, 그 밖의 모든 사정을 참작하여 청구의 취지에 어긋나지 아니하는 범위안에서 사건의 공평한 해결을 위한 화해권고결정을 할 수 있다."
        }
    ]

    # 더 많은 샘플을 생성하기 위한 질문 템플릿
    additional_templates = [
        {
            "question": "불법행위로 인한 손해배상청구권의 소멸시효는 얼마인가요?",
            "answer": "민법 제766조에 따르면, 불법행위로 인한 손해배상청구권은 피해자나 그 법정대리인이 손해 및 가해자를 안 날로부터 3년간 행사하지 않으면 시효로 소멸하고, 불법행위를 한 날로부터 10년이 경과한 때에도 소멸합니다.",
            "context": "불법행위로 인한 손해배상청구권의 단기소멸시효는 피해자의 권리행사 가능성을 고려하여 손해 및 가해자를 안 날로부터 기산합니다."
        },
        {
            "question": "민사소송에서 관할권이란 무엇인가요?",
            "answer": "관할권은 어느 법원이 특정 사건을 심리하고 재판할 권한을 가지는지를 정하는 것입니다. 민사소송법은 토지관할, 사물관할, 심급관할 등을 규정하며, 원칙적으로 피고의 주소지를 관할하는 법원에 소를 제기해야 합니다.",
            "context": "민사소송법은 당사자의 편의와 소송경제를 고려하여 다양한 관할 규정을 두고 있습니다."
        },
        {
            "question": "가압류와 가처분의 차이점은 무엇인가요?",
            "answer": "가압류는 금전채권이나 금전으로 환산할 수 있는 채권의 강제집행을 보전하기 위한 것이고, 가처분은 금전채권 이외의 권리를 보전하기 위한 것입니다. 가압류는 채무자의 재산을 압류하는 것이고, 가처분은 계쟁물에 관한 임시의 지위를 정하는 것입니다.",
            "context": "민사집행법상 가압류와 가처분은 본안판결 전에 권리를 보전하기 위한 임시적 처분입니다."
        }
    ]

    # 전체 샘플 풀 생성
    all_samples = sample_qa_pairs + additional_templates

    # 요청된 수만큼 데이터 생성
    data = []
    for i in range(num_samples):
        qa = all_samples[i % len(all_samples)]

        # 약간의 변형을 추가하여 다양성 확보
        data.append({
            'messages': format_prompt_template(qa['question'], qa.get('context'), qa['answer']),
            'question': qa['question'],
            'answer': qa['answer'],
            'context': qa.get('context', '')
        })

    return data

def preprocess_legal_text(text):
    """
    법률 텍스트 전처리 함수
    - 조항 번호, 날짜, 금액 형식 정규화

    Args:
        text (str): 원본 텍스트

    Returns:
        str: 전처리된 텍스트
    """
    # 조항 번호 정규화 (제 1 조 -> 제1조)
    text = re.sub(r'제\s*(\d+)\s*조', r'제\1조', text)
    text = re.sub(r'제\s*(\d+)\s*항', r'제\1항', text)

    # 날짜 형식 통일 (2024.1.1 -> 2024년 1월 1일)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', r'\1년 \2월 \3일', text)

    # 금액 표기의 쉼표 제거 (1,000,000 -> 1000000)
    text = re.sub(r'(\d{1,3})(,\d{3})+', lambda m: m.group(0).replace(',', ''), text)

    # 연속된 공백을 단일 공백으로 정리
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def extract_legal_entities(text):
    """
    법률 텍스트에서 주요 개체 추출

    Args:
        text (str): 분석할 텍스트

    Returns:
        dict: 추출된 법률 개체들 (법원, 사건번호, 법령)
    """
    return {
        'courts': re.findall(r'[\w]*법원', text),  # 법원명 추출
        'case_numbers': re.findall(r'\d{4}[가-힣]+\d+', text),  # 사건번호 추출
        'laws': re.findall(r'[\w\s]+법(?:\s*제\d+조)?', text)[:5]  # 법령명 추출 (최대 5개)
    }

def format_prompt_template(question, context=None, answer=None):
    """
    Llama 모델용 채팅 형식 프롬프트 템플릿 생성

    Args:
        question (str): 사용자 질문
        context (str, optional): 참고 문서
        answer (str, optional): 정답 (학습시에만 사용)

    Returns:
        list: 메시지 형식의 프롬프트
    """
    # 시스템 프롬프트 설정
    system_message = "당신은 한국 법률 전문가 AI 어시스턴트입니다. 사용자의 법률 질문에 대해 정확하고 전문적인 답변을 제공하세요."

    # 참고 문서가 있는 경우 포함
    if context:
        user_message = f"다음 법률 문서를 참고하여 질문에 답변해주세요.\n\n[참고 문서]\n{context}\n\n[질문]\n{question}"
    else:
        user_message = question

    # 학습용 (답변 포함)
    if answer:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer}
        ]
    # 추론용 (답변 미포함)
    else:
        messages = [
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": [
                    {
                        'type': 'text',
                        'text': user_message
                    }
                ]
            }
        ]

    return messages


# 데이터 로드 및 전처리 실행
print("데이터 전처리 중...")
train_data = create_sample_legal_data(100)
val_data = create_sample_legal_data(10)

# 메모리 및 학습 시간 고려하여 샘플 수 제한
max_train_samples = 1000  # 필요에 따라 조정
max_eval_samples = 100

print(f"전체 학습 데이터: {len(train_data)}")
print(f"전체 검증 데이터: {len(val_data)}")

# 샘플링
train_data = train_data[:max_train_samples]
val_data = val_data[:max_eval_samples]

print(f"전처리 완료!")
print(f"샘플 학습 데이터: {len(train_data)}")
print(f"샘플 검증 데이터: {len(val_data)}")

# 전처리 결과 확인
print("\n전처리된 샘플:")
sample = train_data[0]
print(f"질문: {sample['question']}")
print(f"답변: {sample['answer']}")
print(f"학습 데이터: {sample['messages'][:100]}...")

# HuggingFace Dataset 형식으로 변환
train_dataset = HFDataset.from_list(train_data)
val_dataset = HFDataset.from_list(val_data)

데이터 전처리 중...
전체 학습 데이터: 100
전체 검증 데이터: 10
전처리 완료!
샘플 학습 데이터: 100
샘플 검증 데이터: 10

전처리된 샘플:
질문: 골프 경기 중 선수의 공이 빗나가 다른 사람에게 상해를 입혔을 때 선수에게 요구되는 주의의무는 무엇인가요?
답변: 골프 경기에서 타구를 하는 선수는 자신의 공이 빗나갈 경우를 포함하여 공이 날아갈 것으로 예상되는 범위 내에 사람이 있는지를 확인해야 합니다. 만약 사람이 있는 경우, 목표 방향으로 정확히 보낼 수 있다는 특별한 사정이 없는 이상, 그 사람이 안전한 곳으로 이동할 때까지 기다리거나 공이 사람에게 날아가지 않는 방향으로 타구해야 할 주의의무가 있습니다.
학습 데이터: [{'role': 'system', 'content': '당신은 한국 법률 전문가 AI 어시스턴트입니다. 사용자의 법률 질문에 대해 정확하고 전문적인 답변을 제공하세요.'}, {'role': 'user', 'content': '다음 법률 문서를 참고하여 질문에 답변해주세요.\n\n[참고 문서]\n골프와 같은 개인 운동경기에 참가하는 자는 자신의 행동으로 인해 다른 사람이 다칠 수도 있어서 경기 규칙을 준수하고 주위를 살펴 상해의 결과가 발생하는 것을 미연에 방지해야 할 주의의무가 있습니다.\n\n[질문]\n골프 경기 중 선수의 공이 빗나가 다른 사람에게 상해를 입혔을 때 선수에게 요구되는 주의의무는 무엇인가요?'}, {'role': 'assistant', 'content': '골프 경기에서 타구를 하는 선수는 자신의 공이 빗나갈 경우를 포함하여 공이 날아갈 것으로 예상되는 범위 내에 사람이 있는지를 확인해야 합니다. 만약 사람이 있는 경우, 목표 방향으로 정확히 보낼 수 있다는 특별한 사정이 없는 이상, 그 사람이 안전한 곳으로 이동할 때까지 기다리거나 공이 사람에게 날아가지 않는 방향으로 타구해야 할 주의의무가 있습니다.'}]...


In [4]:
# ============================================
# Cell 3: 모델 로드 및 초기 설정
# ============================================

from trl import SFTConfig, SFTTrainer
from peft import LoraConfig
from transformers import (
    MllamaForConditionalGeneration,
    MllamaProcessor,
    BitsAndBytesConfig
)

# 사용할 한국어 LLM 모델
model_name = "Bllossom/llama-3.2-Korean-Bllossom-AICA-5B"
print(f"선택된 모델: {model_name}")

# 4비트 양자화 설정으로 메모리 효율성 향상
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4비트로 모델 로드
    bnb_4bit_quant_type="nf4",  # Normal Float 4 양자화
    bnb_4bit_compute_dtype=torch.float16,  # 연산은 float16으로
    bnb_4bit_use_double_quant=True,  # 이중 양자화로 추가 압축
)

# 모델 로드 (양자화 적용)
print("모델 로딩 중... (몇 분 소요될 수 있습니다)")
model = MllamaForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",  # GPU 자동 할당
    torch_dtype=torch.float16,  # 메모리 절약을 위한 half precision
    low_cpu_mem_usage=True,  # CPU 메모리 절약
    offload_folder="./offload"   # 디스크 오프로딩
)

# 프로세서 및 토크나이저 설정
processor = MllamaProcessor.from_pretrained(model_name)
tokenizer = processor.tokenizer

# 패딩 토큰 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # 오른쪽 패딩

print("모델 및 토크나이저 로드 완료!")
print(f"모델 크기: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B 파라미터")

# 모델 초기 테스트
test_context = "서울특별시는 대한민국의 수도이며, 인구는 약 950만 명입니다."
test_question = "한국의 수도는 어디인가요?"
template = format_prompt_template(context=test_context, question=test_question)

# 채팅 템플릿 적용
test_prompt = processor.apply_chat_template(
    template,
    tokenize=False,
    add_generation_prompt=True
)

# 토크나이징 및 디바이스 이동
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

print("초기 추론 테스트:")
print(f"질문: {test_question}")

# 추론 실행
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# 생성된 답변 디코딩
response = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"답변: {response}")

선택된 모델: Bllossom/llama-3.2-Korean-Bllossom-AICA-5B
모델 로딩 중... (몇 분 소요될 수 있습니다)


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.58G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

모델 및 토크나이저 로드 완료!
모델 크기: 3.03B 파라미터
초기 추론 테스트:
질문: 한국의 수도는 어디인가요?
답변: 한국의 수도는 서울입니다. 서울은 대한민국의 정치, 경제, 문화의 중심지로, 많은 주요 정부 기관과 기업 본사, 문화 시설 등이 위치해 있습니다.


In [6]:
# ============================================
# Cell 4: LoRA 설정 및 학습 준비
# ============================================

# LoRA (Low-Rank Adaptation) 설정
lora_config = LoraConfig(
    r=8,  # LoRA rank
    lora_alpha=16,  # LoRA scaling parameter
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",  # Attention 모듈
                    "gate_proj", "up_proj", "down_proj"],  # MLP 모듈
    lora_dropout=0.05,  # 드롭아웃 비율
    bias="none",  # 바이어스 학습 안함
    task_type="CAUSAL_LM",  # Causal Language Modeling
)

# SFT (Supervised Fine-Tuning) 학습 설정
training_args = SFTConfig(
    output_dir="./legal-chatbot",  # 모델 저장 경로
    num_train_epochs=1,  # 학습 에폭 수
    per_device_train_batch_size=1,  # 배치 크기
    per_device_eval_batch_size=1,
    gradient_checkpointing=True,     #메모리 절감
    gradient_accumulation_steps=8,  # 그래디언트 누적
    optim="paged_adamw_8bit",  # 최적화 알고리즘
    logging_steps=10,  # 로깅 주기
    logging_first_step=True,
    logging_strategy="steps",
    learning_rate=2e-4,  # 학습률
    warmup_steps=100,  # 웜업 스텝
    save_strategy="steps",  # 저장 전략
    save_steps=200,         # 저장 스텝
    eval_strategy="steps",  # 평가 전략
    eval_steps=50,          # 평가 스텝
    fp16=True,  # 16비트 부동소수점 사용
    max_seq_length=512,  # 최대 시퀀스 길이
    packing=False,  # 시퀀스 패킹 비활성화
    report_to="none",  # 로깅 플랫폼 (wandb 등 사용 가능)
    max_grad_norm=1.0,  # 그래디언트 클리핑
    seed=42,  # 재현성을 위한 시드
    dataloader_num_workers=4,  # 데이터로더 워커 수
    dataloader_pin_memory=True,  # GPU 메모리 고정
    dataset_num_proc=4,  # 데이터셋 전처리 병렬화
    neftune_noise_alpha=5,  # NEFTune 노이즈로 학습 안정성 향상
)

print("학습 설정 완료!")

# 데이터셋 확인
print("학습 데이터셋 최종 샘플 출력")
print(train_dataset[0])
print(val_dataset[0])

# SFTTrainer 초기화
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,  # 토크나이저 전달
    peft_config=lora_config,  # LoRA 설정
)

print("Fine-tuning 시작!")

# 학습 시작
trainer.train()

print("Fine-tuning 완료!")

# 학습된 LoRA 어댑터 저장
model.save_pretrained("./legal-chatbot-lora-adapter")
processor.save_pretrained("./legal-chatbot-lora-adapter")

print("모델 저장 완료!")
print("저장 위치: ./legal-chatbot-lora-adapter")

학습 설정 완료!
학습 데이터셋 최종 샘플 출력
{'messages': [{'content': '당신은 한국 법률 전문가 AI 어시스턴트입니다. 사용자의 법률 질문에 대해 정확하고 전문적인 답변을 제공하세요.', 'role': 'system'}, {'content': '다음 법률 문서를 참고하여 질문에 답변해주세요.\n\n[참고 문서]\n골프와 같은 개인 운동경기에 참가하는 자는 자신의 행동으로 인해 다른 사람이 다칠 수도 있어서 경기 규칙을 준수하고 주위를 살펴 상해의 결과가 발생하는 것을 미연에 방지해야 할 주의의무가 있습니다.\n\n[질문]\n골프 경기 중 선수의 공이 빗나가 다른 사람에게 상해를 입혔을 때 선수에게 요구되는 주의의무는 무엇인가요?', 'role': 'user'}, {'content': '골프 경기에서 타구를 하는 선수는 자신의 공이 빗나갈 경우를 포함하여 공이 날아갈 것으로 예상되는 범위 내에 사람이 있는지를 확인해야 합니다. 만약 사람이 있는 경우, 목표 방향으로 정확히 보낼 수 있다는 특별한 사정이 없는 이상, 그 사람이 안전한 곳으로 이동할 때까지 기다리거나 공이 사람에게 날아가지 않는 방향으로 타구해야 할 주의의무가 있습니다.', 'role': 'assistant'}], 'question': '골프 경기 중 선수의 공이 빗나가 다른 사람에게 상해를 입혔을 때 선수에게 요구되는 주의의무는 무엇인가요?', 'answer': '골프 경기에서 타구를 하는 선수는 자신의 공이 빗나갈 경우를 포함하여 공이 날아갈 것으로 예상되는 범위 내에 사람이 있는지를 확인해야 합니다. 만약 사람이 있는 경우, 목표 방향으로 정확히 보낼 수 있다는 특별한 사정이 없는 이상, 그 사람이 안전한 곳으로 이동할 때까지 기다리거나 공이 사람에게 날아가지 않는 방향으로 타구해야 할 주의의무가 있습니다.', 'context': '골프와 같은 개인 운동경기에 참가하는 자는 자신의 행동으로 인해 다른 사람이 다칠 수도 있어서 경기 규칙을 준수하고 주위를 살펴 상해의 결

Converting train dataset to ChatML (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

...

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Fine-tuning 시작!


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss


Fine-tuning 완료!
모델 저장 완료!
저장 위치: ./legal-chatbot-lora-adapter


In [7]:
# ============================================
# Cell 5: 추론 함수 정의
# ============================================

def generate_legal_response(model, tokenizer, question, context=None, max_length: int = 512):
    """
    법률 질문에 대한 답변 생성 및 후처리

    Args:
        model: 학습된 모델
        tokenizer: 토크나이저
        question (str): 사용자 질문
        context (str, optional): 참고 문서
        max_length (int): 최대 생성 토큰 수

    Returns:
        tuple: (답변 텍스트, 추출된 법률 개체)
    """

    # 입력 텍스트 전처리
    question = preprocess_legal_text(question)
    if context:
        context = preprocess_legal_text(context)

    # 프롬프트 구성
    messages = format_prompt_template(question, context)

    # 채팅 템플릿 적용
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 텍스트 전용 처리 (멀티모달 모델이지만 텍스트만 사용)
    inputs = processor(
        text=prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 생성 설정
    generation_config = {
        'max_new_tokens': max_length,
        'temperature': 0.7,  # 창의성 제어
        'top_p': 0.9,  # nucleus sampling
        'top_k': 50,  # top-k sampling
        'repetition_penalty': 1.1,  # 반복 억제
        'no_repeat_ngram_size': 3,  # n-gram 반복 방지
        'do_sample': True,  # 샘플링 활성화
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
    }

    # 답변 생성
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_config
        )

    # 생성된 부분만 추출하여 디코딩
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # 후처리: 중복 문장 제거
    response = response.strip()
    sentences = response.split('.')
    unique_sentences = []
    for sent in sentences:
        if sent.strip() and sent.strip() not in unique_sentences:
            unique_sentences.append(sent.strip())

    # 문장 재구성
    response = '. '.join(unique_sentences)
    if response and not response.endswith('.'):
        response += '.'

    # 법률 개체 추출
    entities = extract_legal_entities(response)

    # 관련 법령 및 판례 정보 추가
    if entities['laws']:
        response += f"\n\n[관련 법령: {', '.join(entities['laws'][:3])}]"
    if entities['case_numbers']:
        response += f"\n[관련 판례: {', '.join(entities['case_numbers'][:3])}]"

    return response, entities

In [8]:
# ============================================
# Cell 6: 모델 평가 함수 정의 및 실행
# ============================================

from rouge_score import rouge_scorer
from tqdm import tqdm
import numpy as np

def evaluate_legal_chatbot(model, tokenizer, test_data, num_samples=50):
    """
    법률 챗봇 성능 평가
    - ROUGE 스코어 계산
    - 법률 용어 정확도 측정

    Args:
        model: 평가할 모델
        tokenizer: 토크나이저
        test_data: 평가 데이터
        num_samples (int): 평가할 샘플 수

    Returns:
        dict: 평가 메트릭 결과
    """

    # ROUGE 스코어 계산기 초기화
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    results = {'rouge1': [], 'rouge2': [], 'rougeL': [], 'legal_precision': []}

    # 무작위 샘플링
    indices = np.random.choice(len(test_data), min(num_samples, len(test_data)))

    # 각 샘플에 대해 평가
    for idx in tqdm(indices, desc="평가 중"):
        item = test_data[idx]
        question = item['question']
        reference = item['answer']  # 정답

        # 모델 예측 생성
        prediction, pred_entities = generate_legal_response(model, tokenizer, question)

        # ROUGE 점수 계산
        scores = rouge.score(reference, prediction)
        for metric in ['rouge1', 'rouge2', 'rougeL']:
            results[metric].append(scores[metric].fmeasure)

        # 법률 용어 정확도 계산
        ref_entities = extract_legal_entities(reference)
        pred_terms = set(sum(pred_entities.values(), []))
        ref_terms = set(sum(ref_entities.values(), []))

        # 정밀도 계산
        if ref_terms:
            precision = len(pred_terms & ref_terms) / len(ref_terms)
            results['legal_precision'].append(precision)

    return results

print("모델 평가 중...")
# 평가 실행
results = evaluate_legal_chatbot(model, tokenizer, val_data)

# 평가 결과 출력
print("\n=== 평가 결과 ===")
for metric, scores in results.items():
    if scores:
        print(f"{metric}: {np.mean(scores):.4f} (±{np.std(scores):.4f})")

모델 평가 중...


평가 중: 100%|██████████| 10/10 [09:05<00:00, 54.57s/it]


=== 평가 결과 ===
rouge1: 0.0844 (±0.1692)
rouge2: 0.0286 (±0.0857)
rougeL: 0.0844 (±0.1692)
legal_precision: 0.3056 (±0.2437)


In [9]:
# ============================================
# Cell 7: 샘플 테스트 및 메모리 정리
# ============================================

# 테스트할 질문 샘플
questions = [
    "계약 해제 시 손해배상을 청구할 수 있나요?",
    "부당이득반환청구권의 성립 요건은 무엇인가요?",
    "공사대금 채권의 소멸시효는 얼마인가요?"
]

# 모델을 평가 모드로 전환
model.eval()

# 각 질문에 대한 답변 생성
print("\n예측 샘플:")
for question in questions:
    response, entities = generate_legal_response(model, tokenizer, question)
    print(f"\n질문: {question}")
    print(f"답변: {response}")
    if any(entities.values()):
        print(f"추출된 법률 개체: {entities}")

# GPU 메모리 정리
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer

torch.cuda.empty_cache()

print("GPU 메모리 정리 완료!")
print(f"현재 GPU 메모리 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


예측 샘플:

질문: 계약 해제 시 손해배상을 청구할 수 있나요?
답변: 네, 계약 해제 상황에서 손해 배상 청구가 가능합니다. 하지만 이를 위한 조건과 절차는 계약의 성질과 행사 방식에 따라 다를 수 있습니다. 일반적으로 계약 해소 시 손익 관계와 관련된 직접적인 비용이나 손실이 발생한 경우, 이는 보통 합리적인 이유로 인정됩니다. 예를 들어:
1. 계약 당사자 또는 3세자가 불법적이거나 부당한 행위로 인해 발생한 손해. 2. 계약 상의 약속으로 인해 인건비, 장비, 재료 등의 구입, 준비 등이 발생했으나 이에 대한 지불이 실패한 경우. 3. 계약 자체나 일부가 무효화되면서 발생한 경제적 손실. 하지만 계약 해소를 원하는 당사자가 사전에 충분히 협의하지 못한 경우(예: 급하게 계약이 무효가 된 경우), 손해를 배상받기 어려운 경우도 있습니다. 이 외에도 구체적인 상황이나 법률에 따라 달라질 수 있으므로, 특정 상황에 맞는 법적 조언을 원한다면 변호사와 상담하는 것이 좋습니다.

[관련 법령:  계약 당사자 또는 3세자가 불법,  이 외에도 구체적인 상황이나 법,  특정 상황에 맞는 법]
추출된 법률 개체: {'courts': [], 'case_numbers': [], 'laws': [' 계약 당사자 또는 3세자가 불법', ' 이 외에도 구체적인 상황이나 법', ' 특정 상황에 맞는 법']}

질문: 부당이득반환청구권의 성립 요건은 무엇인가요?
답변: 대한민국에서 부당이익(부당계속수입) 환불 청구권이 성립될 수 있는 요건들은 다음과 같습니다:

1. **부당 계속수익 발생**: 부당 계섭수익은 공공기관 또는 기타 공공단체 등이 부당한 방식으로 자금을 배당하는 경우, 해당 자금이 부관직의 직무상 부당하게 발생한 경우에 한해 부당계섭수입으로 인정됩니다. 2. **법률 및 규정**: 이러한 부당 이익 환불 권리가 규정된 법률이나 정부 규정이 필요합니다. 예를 들어, 2017년 11월 10일 법령으로 부당수익에 대한 제도적 허용이나 보상 절차가 명시되어 있습니다. 